# Integración de datos e-Bird

## 1. Definición de parametros

### 1.1 Descripción de variables

Variables del archivo de texto:

|     Variable                 |   Tipo   |      Descripción                                                                                        |
| ---------------------------- | -------- | ------------------------------------------------------------------------------------------------------- |
| *src_txt_file*               |   String | Path de archivo txt a procesar                                                                          |
| *chunk_size*                 |   Number | Tamaño del segmento                                                                                     |
| *src_sep*                    |   String | Separador de campos                                                                                     |

Variables de base de datos:

|     Variable                 |   Tipo   |      Descripción                                                                                  |
| ---------------------------- | -------- | ------------------------------------------------------------------------------------------------- |
| *replace_table_if_exist*     |   Bool   | True o Flase, índica si se sustituye la tabla al intentar la carga                                |
| *table_name*                 |   Str    | Índica el nombre de la tabla, considerar el prefijo (table_prefix, que se estableció como ebird_) |

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from Db_operations import Db_operations
from Txt_data import Txt_data
from Manager import Manager

Db_manager = Db_operations
Manager_user = Manager

#Manager_user.clear_notebook_outputs()  # Borramos las salidas anteriores
chunk_size = 100  # tamaño de la muestra de archivo txt
#src_sep = "\t"  # separador de campos en archivo origen

In [ ]:
Manager_user.clear_notebook_outputs()  # Borramos las salidas anteriores

### 1.2 Definición de variables

Con lo anterior, definimos las variables.

In [ ]:
src_txt_file = replace_table_if_exist = table_name = src_sep = None
src_txt_file = input("Nombre de archivo (considera que debe estár en directorio eBird): ")
src_txt_file = Manager_user.get_file_exists(Manager_user, src_txt_file)

replace_table_if_exist = input("Remplazar la tabla si existe (Posibles valores: True ó False): ")
replace_table_if_exist = Manager_user.get_replace_table_if_exist(Manager_user, replace_table_if_exist)

table_name = (input("Complemento del nombre de tabla (prefijo ebird_): "))

src_sep = input("Delimitador (Posibles valores: tabulador, punto_y_coma, coma y pipe): ")
src_sep = Manager_user.get_delimiter(Manager_user, src_sep)

eval_src_txt_file = Manager_user.check_file_exists(Manager_user, src_txt_file)
eval_replace_table_if_exist = Manager_user.check_replace_table_if_exist(Manager_user, replace_table_if_exist)
eval_table_name = Manager_user.check_table_name(Manager_user, table_name)
eval_src_sep = Manager_user.check_src_sep(Manager_user, src_sep)

gral_eval = Manager_user.get_params_errors(Manager_user, eval_src_txt_file, eval_replace_table_if_exist, eval_table_name, eval_src_sep)

if gral_eval["eval"] == True:
  Manager_user.show_values_selected(Manager_user, src_txt_file, replace_table_if_exist, table_name, src_sep)
else:
  Manager_user.eval_errors(Manager_user, gral_eval["errors"].items())

Procesar el siguiente código únicamente, si se desea utilizar los valores por defecto.

In [ ]:
# Variables del archivo de texto
# src_txt_file = "./../ebd_MX_relJun-2025.txt"  # archivo original, versión completa
src_txt_file = "./../ebd_MX_relJun-2025_first.txt"  # archivo original, versión incompleta
src_sep = "\t"

# Variables de Base de Datos
replace_table_if_exist = True
table_name = "2309"  # consider the prefix with var __table_prefix = "ebird_"

Agregamos los valores para la lectura del archivo.

In [ ]:
Txt_file = Txt_data(src_txt_file, src_sep, chunk_size )

## 2. Creación de la tabla

### 2.1 Subir archivo para definir solucionar los conflictos de campos
El archivo se debe de subir en un directorio por arriba de la ubicación de este archivo y su ubicación se encuentra en la variable *src_txt_file*

### 2.2 Análisis del archivo y generación de SQL de tabla
En base a la lectura del archivo, se genera el SQL sugerido de generación de tabla.

In [ ]:
# Txt_file.get_headers()
txt_fields = Txt_file.get_headers()
sql_new_table = Db_manager.generate_table_sql(Db_manager, txt_fields, table_name)

### 2.3 Creación de la tabla
Ejecutando la siguiente linea, se crea la tabla.

In [ ]:
Db_manager.create_table(Db_manager, table_name, sql_new_table, replace_table_if_exist)

### 2.4 Carga de datos del CSV
Ejecutando la siguiente linea, se sube el CSV.

In [ ]:
result = Db_manager.load_csv_data(Db_manager, src_txt_file, table_name, src_sep, None)

### 2.5 Agregar campos filtro
Ejecutando las siguientes lineas, se agregan las columnas.

In [ ]:
result = Db_manager.create_indx_cols(Db_manager, table_name)  # se crean las columnas
result = Db_manager.set_id_values(
    Db_manager, table_name
)  # agregamos los valores al campo Id
result = Db_manager.add_indx_2_cols(
    Db_manager, table_name
)  # agregamos los índices a las columnas

### 2.6 Depuración
Proceso de la tabla dep.

In [ ]:
Db_manager.drop_dep_table(Db_manager, table_name)  # borra la tabla con prefijo dep_
Db_manager.create_dep_table(Db_manager, table_name)  # crea la tabla con prefijo dep_
Db_manager.add_dep_seinum_index(
    Db_manager, table_name
)  # agrega el índice del campo seinum
Db_manager.populate_dep_table(Db_manager, table_name)  # poblamos la tabla dep
Db_manager.set_gi_num(Db_manager, table_name)  # asiganamos valor la campo gi num
Db_manager.set_dep_mrd(Db_manager, table_name)  # asignamos valor a campo mrd de dep
Db_manager.set_dep_mrd_2(
    Db_manager, table_name
)  # 2da parte de asignació de valor a campo mrd de dep

Proceso de la tabla temporal paso.

In [ ]:
paso_connection = Db_manager.tmp_paso(
    Db_manager, table_name
)  # Proceso de tabla paso. Guardamos la conexión para no perder la tabla temporal paso

### 2.7 Depuración de lista
Creación de índices en tabla taxones_clements_avesmx

In [ ]:
Db_manager.taxo_cle_avmx_add_index(Db_manager)

Actualización de tablas ebird y taxones_clements_avesmx

In [ ]:
Db_manager.update_taxo_cavmx(Db_manager, table_name)

### 2.8 GUI SNIB
Creción de la tabla BUI_SNIB

In [ ]:
# No se procesa
Db_manager.add_gui_snib(Db_manager)

Creación del campo llave

In [ ]:
Db_manager.llave_operations(Db_manager, table_name)

Actualización de **gui_snib**

In [ ]:
Db_manager.update_gui_snib(Db_manager, table_name)

Creación de tabla snib de nuestros datos

In [ ]:
Db_manager.add_snib_version(Db_manager,table_name)

### 2.9 Coordenadas
Creación de la tabla coord

In [ ]:
Db_manager.add_coord(Db_manager, table_name)

Poblamos la tabla

In [ ]:
Db_manager.insert_coord(Db_manager,table_name)

Creamos proceso de tabla temporal coorPaso:

In [ ]:
paso_connection = Db_manager.tmp_coor_paso(Db_manager, paso_connection, table_name)

Creamos proceso de tabla coord_agroup:

In [ ]:
Db_manager.add_coord_agrup(Db_manager, table_name)

In [ ]:
paso_connection = Db_manager.insert_coord_agroup(Db_manager, paso_connection, table_name)

### 2.10 Creación de tabla SNIB
Creación de la tabla ebird_snib***nombre_tabla***


In [ ]:
Db_manager.add_ebird_snib(Db_manager, table_name)